##LogBR Transportes 📦
A LogBR é uma transportadora nacional que opera com múltiplos motoristas e rotas. O time de operações quer analisar o desempenho de entregas para identificar atrasos, comparar motoristas e acompanhar a evolução de cada um ao longo do tempo.

**Problema:** O gestor precisa de um relatório que mostre, para cada entrega:

- Se ela foi mais rápida ou mais lenta que a entrega anterior do mesmo motorista
- A posição do motorista no ranking de pontualidade por região
- O percentual acumulado de entregas no prazo por motorista

## Valores esperados

| Coluna esperada | Descrição |
| :--- | :--- |
| **motorista_nome** | Nome do motorista |
| **regiao** | Região de atuação |
| **entrega_id** | ID da entrega |
| **data_entrega** | Data da entrega |
| **dias_em_transito** | Dias reais |
| **dias_transito_anterior** | dias_em_transito da entrega anterior do mesmo motorista (ordenado por data) |
| **variacao_dias** | Diferença entre a entrega atual e a anterior (negativo = melhorou) |
| **ranking_pontualidade** | Rank do motorista em sua região, baseado no percentual de entregas no prazo — motoristas com mais entregas no prazo ficam melhor rankeados. Use DENSE_RANK. |
| **pct_entregas_no_prazo** | % de entregas no prazo do motorista (sobre o total de entregas dele) |

**Regra de negócio:** `entrega no prazo = dias_em_transito <= prazo_dias`


**Critérios de sucesso** ✅

- dias_transito_anterior deve ser NULL na primeira entrega de cada motorista
- variacao_dias também NULL quando não há entrega anterior
- O ranking deve ser calculado dentro da região, não globalmente
- O pct_entregas_no_prazo deve refletir o histórico completo do motorista, igual para todas as suas linhas



## Artefatos de criação de tabelas
- `sandbox.projetos.motoristas`
- `sandbox.projetos.entregas`


In [0]:
%sql
-- ============================================================
-- TABELA: motoristas
-- ============================================================
CREATE OR REPLACE TABLE sandbox.projetos.motoristas (
  motorista_id   INT     COMMENT 'Identificador único do motorista',
  nome           STRING  COMMENT 'Nome completo',
  regiao         STRING  COMMENT 'Região de atuação: SUL, SUDESTE, NORDESTE'
)
USING DELTA
COMMENT 'Cadastro de motoristas da sandbox.projetos';

-- ============================================================
-- TABELA: entregas
-- ============================================================
CREATE OR REPLACE TABLE sandbox.projetos.entregas (
  entrega_id        INT       COMMENT 'Identificador único da entrega',
  motorista_id      INT       COMMENT 'FK para motoristas',
  data_entrega      DATE      COMMENT 'Data em que a entrega foi realizada',
  prazo_dias        INT       COMMENT 'Prazo contratado em dias corridos',
  dias_em_transito  INT       COMMENT 'Dias reais até a entrega',
  cidade_destino    STRING    COMMENT 'Cidade de destino',
  peso_kg           DECIMAL(8,2) COMMENT 'Peso da carga em kg'
)
USING DELTA
COMMENT 'Registro de entregas realizadas';

In [0]:
%sql
-- ============================================================
-- INSERÇÃO: motoristas (8 motoristas, 3 regiões)
-- ============================================================
INSERT INTO sandbox.projetos.motoristas VALUES
  (1,  'Carlos Souza',    'SUDESTE'),
  (2,  'Ana Lima',        'SUDESTE'),
  (3,  'Roberto Nunes',   'SUL'),
  (4,  'Patrícia Costa',  'SUL'),
  (5,  'Marcos Vieira',   'NORDESTE'),
  (6,  'Juliana Ramos',   'NORDESTE'),
  (7,  'Felipe Duarte',   'SUDESTE'),
  (8,  'Tatiane Melo',    'NORDESTE');

-- ============================================================
-- INSERÇÃO: entregas (30 registros com casos-limite)
-- Casos-limite incluídos:
--   • dias_em_transito = prazo_dias  (entrega exata)
--   • dias_em_transito muito acima do prazo (grande atraso)
--   • datas antigas e recentes
--   • mesmo motorista com várias entregas para análise de janela
-- ============================================================
INSERT INTO sandbox.projetos.entregas VALUES
-- Carlos Souza (id=1) — 5 entregas
  (1,  1, '2024-01-05', 3, 2,  'São Paulo',      120.00),
  (2,  1, '2024-02-10', 3, 4,  'Campinas',        85.50),
  (3,  1, '2024-03-15', 5, 5,  'Rio de Janeiro', 200.00),
  (4,  1, '2024-04-20', 3, 6,  'Santos',          95.00),
  (5,  1, '2024-05-25', 4, 3,  'Sorocaba',        60.00),

-- Ana Lima (id=2) — 4 entregas
  (6,  2, '2024-01-08', 4, 3,  'São Paulo',      310.00),
  (7,  2, '2024-02-14', 4, 5,  'Guarulhos',      150.00),
  (8,  2, '2024-03-20', 3, 3,  'ABC Paulista',    75.00),
  (9,  2, '2024-06-01', 5, 8,  'Ribeirão Preto', 400.00),  -- grande atraso

-- Roberto Nunes (id=3) — 4 entregas
  (10, 3, '2024-01-12', 3, 2,  'Curitiba',       180.00),
  (11, 3, '2024-02-18', 3, 3,  'Florianópolis',  220.00),
  (12, 3, '2024-04-05', 4, 4,  'Porto Alegre',   135.00),
  (13, 3, '2024-05-10', 3, 2,  'Blumenau',        90.00),

-- Patrícia Costa (id=4) — 3 entregas
  (14, 4, '2024-02-22', 5, 4,  'Porto Alegre',   500.00),
  (15, 4, '2024-03-30', 3, 3,  'Curitiba',       210.00),
  (16, 4, '2024-05-15', 4, 6,  'Joinville',      175.00),  -- atraso

-- Marcos Vieira (id=5) — 4 entregas
  (17, 5, '2024-01-20', 4, 3,  'Fortaleza',      130.00),
  (18, 5, '2024-03-08', 3, 4,  'Recife',         200.00),  -- atraso
  (19, 5, '2024-04-14', 5, 5,  'Salvador',       310.00),  -- exato
  (20, 5, '2024-06-10', 3, 2,  'Natal',           95.00),

-- Juliana Ramos (id=6) — 4 entregas
  (21, 6, '2024-01-25', 3, 2,  'Recife',         160.00),
  (22, 6, '2024-02-28', 4, 4,  'Fortaleza',      240.00),  -- exato
  (23, 6, '2024-04-18', 3, 3,  'Maceió',         190.00),
  (24, 6, '2024-05-30', 5, 4,  'João Pessoa',    115.00),

-- Felipe Duarte (id=7) — 3 entregas
  (25, 7, '2024-03-05', 3, 5,  'São Paulo',      280.00),  -- atraso
  (26, 7, '2024-04-12', 4, 3,  'Campinas',       165.00),
  (27, 7, '2024-06-20', 3, 2,  'São Paulo',      320.00),

-- Tatiane Melo (id=8) — 3 entregas
  (28, 8, '2024-02-05', 4, 3,  'Salvador',       145.00),
  (29, 8, '2024-04-22', 3, 3,  'Fortaleza',      230.00),  -- exato
  (30, 8, '2024-06-15', 5, 7,  'Recife',         410.00);  -- grande atraso

In [0]:
%sql
select
m.nome as motorista_nome,
m.regiao,
e.data_entrega,
e.prazo_dias,
e.dias_em_transito,
lag(e.dias_em_transito,1) over (partition by m.motorista_id order by e.data_entrega) as dias_em_transito_anterior,
e.dias_em_transito - lag(e.dias_em_transito,1) over (partition by m.motorista_id order by e.data_entrega) as variacao_dias,
round(sum(if(e.dias_em_transito <= e.prazo_dias,1,0)) over (partition by m.motorista_id) / count(*) over (partition by m.motorista_id),2) 
as pct_entregas_em_dias
from sandbox.projetos.motoristas as m
join sandbox.projetos.entregas as e
on m.motorista_id = e.motorista_id


    

